현재 legacy로 넘어가서 권장 방식이 바뀐 사용법이 많다.

| 기존                            | 현재 신규 코드에서 추천                      | 핵심        |
| ----------------------------- | ---------------------------------- | --------- |
| `ResponseSchema`              | `Pydantic BaseModel + Field`       | Schema 정의 |
| `StructuredOutputParser`      | `with_structured_output()`         | 구조화 출력    |
| `DatetimeOutputParser`        | Pydantic `datetime`                | 날짜 타입 검증  |
| `EnumOutputParser`            | Pydantic + `Enum`                  | 선택지 제한    |
| `PandasDataFrameOutputParser` | Structured Output + Pandas 코드/Tool | 데이터 처리    |

LangChain 문서에서도 Pydantic을 structured output의 schema로 지원하며, field validation, description, nested structure 등의 장점을 제공합니다.

과거 LangChain
             LLM
              │
              ▼
           String
              │
       ┌──────┴───────┐
       │              │
DatetimeParser     EnumParser
StructuredParser   PandasParser
       │              │
       ▼              ▼
   Python Object    Result

즉 LLM은 문자열을 생성하고 LangChain Parser가 해석했습니다.

현재 LangChain
                LLM
                 │
        Structured Output
                 │
                 ▼
        ┌─────────────────┐
        │ Pydantic Schema │
        └─────────────────┘
                 │
        ┌────────┼─────────┐
        ▼        ▼         ▼
      Enum    datetime   List/객체
                 │
                 ▼
          Application
                 │
         ┌───────┴───────┐
         ▼               ▼
       Pandas           Tools

LangChain 공식 문서에서도 현재 with_structured_output()에 Pydantic / TypedDict / JSON Schema를 전달하는 방식을 지원하며, Pydantic은 validation과 field description, nested structure 등을 지원합니다.

Agent까지 가면 한 단계 더 발전해서 response_format=Schema를 전달할 수 있고, provider가 native structured output을 지원하면 LangChain이 ProviderStrategy를 사용할 수 있습니다. 그렇지 않은 경우 tool-calling 전략을 사용할 수 있습니다.

그래서 지금 수업을 들으면서 외우실 것은 각 Parser의 API가 아니라 다음 대응 관계입니다.

ResponseSchema
        ┐
StructuredOutputParser ──→ Pydantic + with_structured_output()
        │
DatetimeOutputParser   ──→ datetime field
        │
EnumOutputParser       ──→ Enum field
        │
PandasDataFrameParser  ──→ Structured Output + Python/Pandas/Tool

그리고 StrOutputParser, JsonOutputParser, PydanticOutputParser는 조금 다르게 봐야 합니다. 이 셋은 무조건 "레거시니까 버린다"라고 분류하면 안 됩니다. 다음 Output Parser 정리를 하실 때 이 셋까지 포함해서 **2026 LangChain v1 기준 계속 사용 / 상황에 따라 사용 / Classic이라 신규 코드 비추천**으로 나누면 Output Parser 파트가 깔끔하게 끝납니다.

In [1]:
import pprint
from typing import Any, Dict
import pandas as pd
from langchain_classic.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-Current-Output-Parser")

model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-Current-Output-Parser


#### ResponseSchema + StructuredOutputParser -> pydantic + with_structured_output()
ResponseSchema
        ↓
StructuredOutputParser
        ↓
get_format_instructions()
        ↓
Prompt
        ↓
LLM
        ↓
parse()

          ⬇️

Pydantic BaseModel
        ↓
with_structured_output()
        ↓
LLM
        ↓
Pydantic Object

In [ ]:
from pydantic import BaseModel, Field
class Person(BaseModel):
    name: str = Field(description="사용자의 이름")
    age: int = Field(description="사용자의 나이")
    
structured_llm = model.with_structured_output(Person)

result = structured_llm.invoke("김철수는 25살 입니다.")
print(result)

name='김철수' age=25


#### 마찬가지로 Datetime output parser를 사용하는대신 pydantic + wit_structured_output으로 사용한다. 

In [6]:
from datetime import datetime
from pydantic import BaseModel, Field

class Schedule(BaseModel):
    event: str
    start_time: datetime = Field(description="일정 시작 시간")
    
structured_llm = model.with_structured_output(Schedule)

result = structured_llm.invoke("AI 회의는 2026년 09월 20일 오후 3시에 진행됩니다.")

print(result)
print(result.start_time)
print(type(result.start_time))

event='AI 회의' start_time=datetime.datetime(2026, 9, 20, 15, 0, tzinfo=TzInfo(0))
2026-09-20 15:00:00+00:00
<class 'datetime.datetime'>


In [12]:
from enum import Enum
from pydantic import BaseModel

class Priority(str, Enum):
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"
    
class TaskResult(BaseModel):
    priority: Priority 
    
class TaskResult_EXP(BaseModel):
    priority: Priority 
    reason: str
    confidence: float
    
structured_llm = model.with_structured_output(TaskResult)

situation = """ 현재 서버가 다운되서 게임 유저들의 컴플레인이 빗발치고 있다."""


result = structured_llm.invoke(situation)
print(result)

structured_llm_exp = model.with_structured_output(TaskResult_EXP)
result_exp = structured_llm_exp.invoke(situation)
print(result_exp)

priority=<Priority.HIGH: 'high'>
priority=<Priority.HIGH: 'high'> reason='서버 다운으로 인해 유저들이 게임을 이용할 수 없고, 이는 유저 경험에 큰 영향을 미치며, 유저 이탈로 이어질 수 있습니다. 빠른 해결이 필요합니다.' confidence=0.9


In [ ]:
from pydantic import BaseModel
import pandas as pd
'''
from langchain_classic.output_parsers import (
    PandasDataFrameOutputParser
)

parser = PandasDataFrameOutputParser(
    dataframe=df
)
둘다 사용
'''
# person class 와 people class + with_structured_output으로 Pandas output 구현 
class Person(BaseModel):
    name: str
    age: int
    score: float


class People(BaseModel):
    rows: list[Person]


structured_llm = model.with_structured_output(People)

result = structured_llm.invoke(
    """
    철수 25살 90점
    영희 27살 95점
    민수 24살 85점
    """
)

df = pd.DataFrame([
    row.model_dump()
    for row in result.rows
])

print(df)

  name  age  score
0   철수   25   90.0
1   영희   27   95.0
2   민수   24   85.0
